ecommerce\src\user_app\serializers.py

In [ ]:
from django.contrib.auth.models import User
from rest_framework.serializers import ModelSerializer, ValidationError, CharField


class RegistrationSerializer(ModelSerializer):
    password2 = CharField(
        style={"input_type": "password"}, write_only=True
    )

    class Meta:
        model = User
        fields = ["username", "email", "password", "password2"]
        extra_kwargs = {
            "password": {"write_only": True}
        }

    def save(self):
        password = self.validated_data["password"]
        password2 = self.validated_data["password2"]

        if password != password2:
            raise ValidationError({"error": "password y password2 deben ser iguales"})

        email = self.validated_data["email"]

        if User.objects.filter(email=email).exists():
            raise ValidationError({"error": "Ese correo ya está registrado"})

        username = self.validated_data["username"]

        account = User(email=email, username=username)
        account.set_password(password)
        account.save()

        return account


class UserProfileSerializer(ModelSerializer):
    class Meta:
        model = User
        fields = ["id", "username", "email", "date_joined"]

ecommerce\src\user_app\views.py

In [ ]:
from rest_framework.decorators import api_view
from rest_framework.response import Response
from rest_framework import status
from rest_framework_simplejwt.tokens import RefreshToken

from rest_framework.views import APIView
from rest_framework.permissions import IsAuthenticated
from rest_framework_simplejwt.authentication import JWTAuthentication

from user_app.serializers import RegistrationSerializer, UserProfileSerializer


@api_view(["POST"])
def registration_view(request):
    if request.method == "POST":
        serializer = RegistrationSerializer(data=request.data)

        data = {}

        if serializer.is_valid():
            account = serializer.save()

            data["response"] = "Registro Exitoso"
            data["username"] = account.username
            data["email"] = account.email

            refresh_token = RefreshToken.for_user(account)
            data["token"] = {
                "refresh": str(refresh_token),
                "access": str(refresh_token.access_token)
            }
        else:
            data = serializer.errors

        return Response(data)


class UserProfileView(APIView):
    authentication_classes = [JWTAuthentication]
    permission_classes = [IsAuthenticated]

    def get(self, request):
        serializer = UserProfileSerializer(request.user)
        return Response(serializer.data, status=status.HTTP_200_OK)

ecommerce\src\user_app\urls.py

In [ ]:
from django.urls import path
from rest_framework_simplejwt.views import TokenObtainPairView, TokenRefreshView

from user_app.views import registration_view, UserProfileView

urlpatterns = [
    path("register/", registration_view),
    path("api/token/", TokenObtainPairView.as_view()),
    path("api/token/refresh/", TokenRefreshView.as_view()),

    path("profile/", UserProfileView.as_view()),
]